Risk Classifcation 

In [1]:
!pip install xgboost lightgbm catboost scikit-learn pandas matplotlib seaborn joblib 

   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ------- -------------------------------- 0.3/1.5 MB ? eta -:--:--
   --------------------- ------------------ 0.8/1.5 MB 2.4 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 2.6 MB/s  0:00:00
   ---------------------------------------- 0.0/100.2 MB ? eta -:--:--
   ---------------------------------------- 0.5/100.2 MB 3.1 MB/s eta 0:00:33
    --------------------------------------- 1.3/100.2 MB 3.3 MB/s eta 0:00:30
    --------------------------------------- 1.8/100.2 MB 3.3 MB/s eta 0:00:30
   - -------------------------------------- 2.6/100.2 MB 3.4 MB/s eta 0:00:29
   - -------------------------------------- 3.4/100.2 MB 3.4 MB/s eta 0:00:29
   - -------------------------------------- 4.2/100.2 MB 3.5 MB/s eta 0:00:28
   - -------------------------------------- 4.7/100.2 MB 3.4 MB/s eta 0:00:28
   -- ------------------------------------- 5.5/100.2 MB 3.4 MB/s eta 0:00:28
   -- ------------


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Imports of all needed data, files and modules


In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix,
                              accuracy_score, f1_score)
import matplotlib.pyplot as plt
import seaborn as sns

In [3]:
df = pd.read_csv("Diabetes_and_Lifestyle_Dataset_.csv")

# Quick inspection
print(df.shape)
print(df['diabetes_stage'].value_counts())
print(df.isnull().sum())
le = LabelEncoder()
df['diabetes_stage_encoded'] = le.fit_transform(df['diabetes_stage'])
X = df.drop(columns=['diabetes_stage', 'diabetes_stage_encoded'])
y = df['diabetes_stage_encoded']
# Encode any remaining categorical columns
X = pd.get_dummies(X)

(97297, 31)
diabetes_stage
Type 2          58163
Pre-Diabetes    31013
No Diabetes      7737
Gestational       267
Type 1            117
Name: count, dtype: int64
Age                                   0
gender                                0
ethnicity                             0
education_level                       0
income_level                          0
employment_status                     0
smoking_status                        0
alcohol_consumption_per_week          0
physical_activity_minutes_per_week    0
diet_score                            0
sleep_hours_per_day                   0
screen_time_hours_per_day             0
family_history_diabetes               0
hypertension_history                  0
cardiovascular_history                0
bmi                                   0
waist_to_hip_ratio                    0
systolic_bp                           0
diastolic_bp                          0
heart_rate                            0
cholesterol_total                    

In [6]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
y = le.fit_transform(y)

for col in X.select_dtypes(include='object').columns:
    X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    


In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
le = LabelEncoder()
y = le.fit_transform(df['diabetes_stage'])
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print("Class mapping:", dict(zip(le.classes_, le.transform(le.classes_))))

Train: (77837, 48), Test: (19460, 48)
Class mapping: {'Gestational': np.int64(0), 'No Diabetes': np.int64(1), 'Pre-Diabetes': np.int64(2), 'Type 1': np.int64(3), 'Type 2': np.int64(4)}


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 80% train, 20% test
    random_state=42,     # reproducibility
    stratify=y           # keeps class proportions in both splits
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples:     {X_test.shape[0]}")

Training samples: 77837
Test samples:     19460


In [10]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

# Confirm it balanced the classes
print(pd.Series(y_train_bal).value_counts())

2    46530
1    46530
4    46530
0    46530
3    46530
Name: count, dtype: int64


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled  = scaler.transform(X_test)  

Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

dt = DecisionTreeClassifier(
    max_depth=5,          # controls how deep the tree grows
    min_samples_split=10, # min samples needed to split a node
    random_state=42
)
le = LabelEncoder()
y = le.fit_transform(df['diabetes_stage'])
X = df.drop(columns=['diabetes_stage'])
dt.fit(X_train_bal, y_train_bal)       # train
y_pred_dt = dt.predict(X_test)        # predict on test set

print("DECISION TREE RESULTS")
print(classification_report(y_test, y_pred_dt, target_names=le.classes_))


 Tuned Decision Tree

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.tree import DecisionTreeClassifier

dt_params = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 5],
    'criterion': ['gini', 'entropy']
}

dt_grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    dt_params,
    cv=5,                    # 5-fold cross validation
    scoring='f1_weighted',   # optimise for weighted F1
    n_jobs=-1,
    verbose=1
)

dt_grid.fit(X_train_bal, y_train_bal)

print("Best DT params:", dt_grid.best_params_)
print("Best DT score: ", dt_grid.best_score_)

# Use the best version
best_dt = dt_grid.best_estimator_

Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=100,    # number of trees
    max_depth=10,        # max depth per tree
    random_state=42,
    n_jobs=-1            # use all CPU cores
)

rf.fit(X_train_bal, y_train_bal)
y_pred_rf = rf.predict(X_test)

print("RANDOM FOREST RESULTS")
print(classification_report(y_test, y_pred_rf, target_names=le.classes_))

Tuned Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'max_features': ['sqrt', 'log2']
}

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_params,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_bal, y_train_bal)

print("Best RF params:", rf_grid.best_params_)
best_rf = rf_grid.best_estimator_

Xgboost

In [ ]:
from xgboost import XGBClassifier

# Count classes for scale_pos_weight (handles imbalance further)
xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,   # how much each tree contributes
    max_depth=6,
    subsample=0.8,        # fraction of data per tree
    colsample_bytree=0.8, # fraction of features per tree
    random_state=42,
    eval_metric='mlogloss',
    use_label_encoder=False
)

xgb.fit(
    X_train_scaled, y_train_bal,
    eval_set=[(X_test_scaled, y_test)],   # monitor test performance
    verbose=50                             # print every 50 rounds
)

from xgboost import XGBClassifier

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='mlogloss', 
                  use_label_encoder=False),
    xgb_params,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_bal, y_train_bal)

print("Best XGB params:", xgb_grid.best_params_)
best_xgb = xgb_grid.best_estimator_
y_pred_xgb = xgb.predict(X_test_scaled)

print("XGBOOST RESULTS")
print(classification_report(y_test, y_pred_xgb, target_names=le.classes_))


In [ ]:
from xgboost import XGBClassifier

xgb_params = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [3, 5, 7],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

xgb_grid = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='mlogloss', 
                  use_label_encoder=False),
    xgb_params,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)

xgb_grid.fit(X_train_bal, y_train_bal)

print("Best XGB params:", xgb_grid.best_params_)
best_xgb = xgb_grid.best_estimator_

Comparison Between Models

In [ ]:
from sklearn.model_selection import cross_val_score

for name, model in [('Decision Tree', best_dt), 
                     ('Random Forest', best_rf), 
                     ('XGBoost', best_xgb)]:
    
    scores = cross_val_score(model, X_train_bal, y_train_bal, 
                             cv=5, scoring='f1_weighted')
    
    print(f"{name}: {scores.mean():.4f} (+/- {scores.std():.4f})")
    

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

models = {
    'Decision Tree': y_pred_dt,
    'Random Forest': y_pred_rf,
    'XGBoost':       y_pred_xgb
}

print(f"{'Model':<20} {'Accuracy':>10} {'F1 (weighted)':>15}")
print("-" * 47)
for name, preds in models.items():
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='weighted')
    print(f"{name:<20} {acc:>10.4f} {f1:>15.4f}")
    results_df = pd.DataFrame(results).T
print(results_df)

results_df.plot(kind='bar', figsize=(8, 5), colormap='Set2')
plt.title('Model Comparison')
plt.ylabel('Score')
plt.xticks(rotation=0)
plt.ylim(0, 1)
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('model_comparison.png', dpi=150)
plt.show()

In [ ]:
tuned_models = {
    'Decision Tree (tuned)': (best_dt, X_test),
    'Random Forest (tuned)': (best_rf, X_test),
    'XGBoost (tuned)':       (best_xgb, X_test_scaled)}

print(f"{'Model':<25} {'Accuracy':>10} {'F1 Weighted':>13}")
print("-" * 50)
for name, (model, X_eval) in tuned_models.items():
    preds = model.predict(X_eval)
    acc = accuracy_score(y_test, preds)
    f1  = f1_score(y_test, preds, average='weighted')
    print(f"{name:<25} {acc:>10.4f} {f1:>13.4f}")

Exporting Models

In [ ]:
import joblib

joblib.dump(xgb,    'models/best_classifier.pkl')   # swap for whichever won
joblib.dump(le,     'models/label_encoder.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(list(X.columns), 'models/feature_columns.pkl')
